### Importing libraries

In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

### Importing data

In [3]:
currency_index= pd.read_csv("C:/Users/user/Desktop/Data Science/PoeMarketAnalyser/data/poeninja/currency_data_processed/currency_index.csv", sep=";", low_memory=False)
players = pd.read_csv("C:/Users/user/Desktop/Data Science/PoeMarketAnalyser/data/poeninja/players_data_processed/players_processed.csv", sep=";", low_memory=False)

# Players activity ecnomics

### Adding **Day** Column

In [4]:
#first day of each league
players["Date"] = pd.to_datetime(players["Date"])
league_start = players.groupby("League")["Date"].transform("min")

# league day
players["Day"] = (players["Date"]-league_start).dt.days + 1
players_day=players.drop(columns=["Date"]).copy()

#exporting players dataset with Day column
players_day.reset_index(drop=True).to_csv("C:/Users/user/Desktop/Data Science/PoeMarketAnalyser/data/poeninja/players_data_processed/players_processed_day.csv",sep=";",index=False)

### Merging datasets

In [5]:
economy_population = currency_index.merge(players_day,on=["League", "Day"],how="left")
economy_population.head()

,League,Day,currency_index,Players
0,Affliction,1,1.000000,144193.0
1,Affliction,2,0.687330,144326.0
2,Affliction,3,0.954545,124599.0
3,Affliction,4,1.128216,116564.0
4,Affliction,5,1.357263,109595.0


### Players number normalisation
As the number of players differ from leagues

In [6]:
# player population on day 1 
day1_players = (economy_population.sort_values(["League", "Day"]).groupby("League", as_index=False).first()[["League", "Players"]].rename(columns={"Players": "Players_day1"}))

# Merge population with day 1 players
economy_population = economy_population.merge(day1_players,on="League",how="left")

# Number of players normalisation
economy_population["Players_norm"] = (economy_population["Players"] /economy_population["Players_day1"])
economy_population.head()

,League,Day,currency_index,Players,Players_day1,Players_norm
0,Affliction,1,1.000000,144193.0,144193.0,1.000000
1,Affliction,2,0.687330,144326.0,144193.0,1.000922
2,Affliction,3,0.954545,124599.0,144193.0,0.864113
3,Affliction,4,1.128216,116564.0,144193.0,0.808389
4,Affliction,5,1.357263,109595.0,144193.0,0.760058


### Correlation between number of players and Currency Purchasing Index 

To examine the relationship between player population and inflation in the league economy, two correlation measures were used: Pearson and Spearman.

Pearson correlation measures the strength of a linear relationship between two variables. In this context, it indicates whether changes in player population are proportionally associated with changes in the currency inflation index. Because Spearman correlation measures a monotonic relationship based on the ranking of values, it is more robust to outliers, non-linear relationships, and short-term fluctuations in the data.

Using both measures increases the reliability of the analysis. If Pearson and Spearman produce similar results, it suggests that the observed relationship is consistent and not driven by extreme observations or non-linear behaviour.

In [7]:
# Pearson correlation 
pearson_corr = (economy_population.groupby("League")[["Players_norm", "currency_index"]].corr(method="pearson").unstack().iloc[:,1].reset_index(name="pearson_corr"))

# Spearman correlation 
spearman_corr = (economy_population.groupby("League")[["Players_norm", "currency_index"]].corr(method="spearman").unstack().iloc[:,1].reset_index(name="spearman_corr"))

# Merging results
correlation_summary = pearson_corr.merge(spearman_corr,on="League")
correlation_summary

,League,pearson_corr,spearman_corr
0,Affliction,-0.607559,-0.510147
1,Ancestor,-0.926210,-0.970830
2,Keepers,-0.542178,-0.426656
3,Mercenaries,-0.361334,-0.018029
4,Necropolis,-0.100340,0.405536
5,Settlers,-0.669447,-0.744255


In [13]:
fig = px.scatter( economy_population,x="Players_norm", y="currency_index",color="League",title="Player population vs currency inflation",)
fig.update_layout(xaxis_title="Normalised player population",yaxis_title="Currency purchasing power index")
fig.show()

**Conclusions:**

1. The comparison of average inflation rates shows substantial differences between leagues. The **Ancestor** league exhibits the highest inflation levels, with weekly inflation close to 8%. In real-world economic terms, inflation at this level would be considered very high, indicating a rapid decline in the purchasing power of the currency.

2. The **Keepers** and the **Affliction** leagues also show high inflation level, while **Necropolis** and **Mercenaries** leagues show lower inflation levels, with weekly values around 4%. This range is considered as moderate inflation and suggests a relatively stable currency market.

3. The **Settlers** league exhibits the lowest inflation among the analysed leagues, with weekly inflation close to 2.5%. In real-world economics, inflation around 2-2.5% is considered ideal, low, and stable, as it allows economic activity to grow, avoiding deflation at the same time. The **Settlers** league can therefore be considered the closest representation of an economically stable market. 

4. The daily and weekly measures follow the same relative ordering across leagues, suggesting that the observed differences reflect persistent economic characteristics of each league rather than short-term market fluctuations.
